# Week 11: Daubechies wavelets and the cascade algorithm

**What we do.** We construct the Daubechies filters ourselves by spectral factorization (as in the lecture notes), draw the scaling function `φ` and the wavelet `ψ` with the cascade algorithm, watch the cascade iterations converge, and split a signal into equal frequency bands with wavelet packets.

**What to expect.** Our filters coincide with those of PyWavelets. `φ` and `ψ` of db1 (Haar) are discontinuous; with more vanishing moments (`p = 2, 3, 5`) they become longer and smoother. Wavelet packets locate the two frequencies of a test signal in two of eight bands.

**How to read the code.** Functions come from `wavelets_lab.mra`. `p` is the number of vanishing moments; the filter `db{p}` has `2p` coefficients and `φ`, `ψ` are supported on `[0, 2p - 1]`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelets_lab import mra

## Daubechies filters by spectral factorization

**What we do.** We compute the filters for `p = 1, …, 4` and compare them with PyWavelets.

**What to expect.** For `p = 1` the Haar filter `[0.707, 0.707]`; for larger `p` longer filters with some negative coefficients. The largest difference from PyWavelets is about `1e-15`.

**How to read the code.** `mra.daubechies_filter(p)` builds the polynomial `P` of the lecture notes, finds its roots with `np.roots`, keeps those inside the unit circle and multiplies out the filter. `mra.wavelet_filters(f"db{p}")[0]` is the PyWavelets low-pass filter.

In [ ]:
for p in range(1, 5):
    h = mra.daubechies_filter(p)
    difference = np.abs(h - mra.wavelet_filters(f"db{p}")[0]).max()
    print(f"p = {p}: {np.round(h, 6)}  difference from pywt: {difference:.1e}")

## The filter coefficients

**What we do.** We plot the coefficients of the db4 filter `h` and of its wavelet filter `g`.

**What to expect.** Eight coefficients each; `h` is mostly positive (an averaging filter, `Σ h = sqrt 2`), `g` alternates in sign (a differencing filter, `Σ g = 0`).

**How to read the code.** `plt.stem` draws one vertical line per coefficient; `mra.quadrature_mirror(h)` gives `g[n] = (-1)^n h[L - 1 - n]`.

In [ ]:
h = mra.daubechies_filter(4)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].stem(h)
axes[0].set_title("low-pass filter h (db4)")
axes[1].stem(mra.quadrature_mirror(h))
axes[1].set_title("high-pass filter g (db4)")
for ax in axes:
    ax.set_xlabel("index n")
    ax.set_ylabel("coefficient")
plt.tight_layout()
plt.show()

## The cascade algorithm

**What we do.** We draw `φ` and `ψ` for db1, db2, db3 and db5 after 8 iterations of the cascade algorithm.

**What to expect.** db1: the indicator of `[0, 1)` and the Haar wavelet. db2: a fractal-looking, continuous but not differentiable curve on `[0, 3]`. db3 and db5: longer and smoother. `ψ` always oscillates (its integral is 0).

**How to read the code.** `mra.cascade(h, iterations=8)` returns the grid `x = m / 256` and the values of `φ` and `ψ` on it. `axes[row, column]` addresses the panel in a 2×4 grid.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for column, p in enumerate((1, 2, 3, 5)):
    x, phi, psi = mra.cascade(mra.daubechies_filter(p), iterations=8)
    axes[0, column].plot(x, phi)
    axes[0, column].set_title(f"scaling function φ, db{p}")
    axes[1, column].plot(x, psi)
    axes[1, column].set_title(f"wavelet ψ, db{p}")
for ax in axes.flat:
    ax.set_xlabel("x")
    ax.set_ylabel("value")
plt.tight_layout()
plt.show()

## Convergence of the iterations

**What we do.** We run the cascade algorithm for db2 with 1, 2, 3 and 6 iterations.

**What to expect.** After one iteration a coarse broken line with 4 points; every iteration doubles the resolution and the curves settle down to the limit function `φ` (after 6 iterations it is visually final).

**How to read the code.** `mra.cascade` returns values on the grid `x = m / 2^iterations`; `"o-"` marks the computed points, which shows the resolution of each iteration.

In [ ]:
plt.figure(figsize=(9, 4))
for iterations in (1, 2, 3, 6):
    x, phi, _ = mra.cascade(mra.daubechies_filter(2), iterations)
    plt.plot(x, phi, "o-", markersize=3, label=f"{iterations} iterations")
plt.xlabel("x")
plt.ylabel("φ(x) for db2")
plt.legend()
plt.show()

## Properties of φ and ψ

**What we do.** We check numerically that `∫ φ = 1` and `∫ ψ = 0` for db3.

**What to expect.** The two printed numbers are close to 1 and 0. (`∫ φ = 1` follows from the normalization `Σ h = sqrt 2`, and `∫ ψ = 0` from `Σ g = 0`.)

**How to read the code.** On the grid with step `x[1] - x[0]` an integral is approximated by `Σ values × step`.

In [ ]:
x, phi, psi = mra.cascade(mra.daubechies_filter(3), iterations=10)
step = x[1] - x[0]
np.sum(phi) * step, np.sum(psi) * step

## Wavelet packets

**What we do.** We decompose a signal containing two sinusoids (32 Hz and 288 Hz, sampled at 1024 Hz) into 8 equal frequency bands with wavelet packets, once with db4 and once with the longer db8, and plot the energy of each band.

**What to expect.** The ordinary DWT splits only the low frequencies again and again; packets split every band, giving eight bands of width `512 / 8 = 64` Hz. The energy concentrates in band 0 (0–64 Hz, containing 32 Hz) and band 4 (256–320 Hz, containing 288 Hz). The filters are not ideal, so part of the energy of the 288 Hz tone leaks into band 3 (192–256 Hz); the longer db8 filters leak less and are cleaner elsewhere.

**How to read the code.** `mra.packet_energies(signal, wavelet, 3)` returns a dictionary `{path: energy}` in frequency order; a path like `"aad"` records the sequence of low-pass (`a`) and high-pass (`d`) filters (the last line prints the paths in frequency order). Bars of the two wavelets are drawn side by side (shifted by ±0.2) at the band index; the tick labels give the frequency range of each band.

In [ ]:
t = np.arange(1024) / 1024
signal = np.sin(2 * np.pi * 32 * t) + np.sin(2 * np.pi * 288 * t)
band = np.arange(8)
plt.figure(figsize=(10, 4))
for shift, name in ((-0.2, "db4"), (0.2, "db8")):
    energies = mra.packet_energies(signal, name, 3)
    plt.bar(band + shift, list(energies.values()), width=0.4, label=name)
plt.xticks(band, [f"{64 * k}-{64 * (k + 1)}" for k in band])
plt.xlabel("frequency band [Hz]")
plt.ylabel("energy")
plt.legend()
plt.show()
list(mra.packet_energies(signal, "db4", 3))

## Reference: scripts and functions used in this notebook

**What we do.** We list every function of the course package used above with its purpose, input, output and a recipe for writing it yourself.

**What to expect.** One module is used: `wavelets_lab.mra` (`src/wavelets_lab/mra.py`).

**How to read the code.** `p` is the number of vanishing moments and the filter `db{p}` has length `2p`.

**Library functions called directly in this notebook.**

| Function | What it does | Input → output |
|---|---|---|
| `plt.stem(values)` | vertical lines at the positions `0, 1, …` | array → plot |
| `np.round(a, 6)` | rounds to 6 decimals for printing | array → array |
| `np.sum(values) * step` | Riemann sum of a sampled function | array → number |
| `plt.bar(labels, heights)` | bar chart | lists → bars |

### Module `wavelets_lab.mra` (file `src/wavelets_lab/mra.py`)

Multiresolution analysis (MRA): filters, the construction of Daubechies wavelets and the cascade algorithm. A **low-pass filter** `h` is the sequence of the scaling equation `φ(x) = sqrt(2) sum_n h[n] φ(2x - n)`, normalized so that `sum_n h[n] = sqrt(2)`; this is the `rec_lo` filter of PyWavelets. Its frequency response is `m0(ω) = (1/sqrt 2) sum_n h[n] e^{-inω}`. Imports:
```python
from math import comb
import numpy as np
import pywt
```
and the constant `SQRT2 = sqrt(2)`.

#### `mra.daubechies_filter(p)`

**What it is.** Constructs the Daubechies filter with `p` vanishing moments (length `2p`) by spectral factorization, as in the lecture notes. Daubechies proved that
`|m0(ω)|^2 = cos^{2p}(ω/2) P(sin^2(ω/2))` with `P(y) = sum_{k=0}^{p-1} C(p-1+k, k) y^k`.
Writing `z = e^{-iω}`, the factor `cos^{2p}(ω/2)` gives `p` roots at `z = -1` of `m0`, and `P` (as a polynomial in `z`) has roots in pairs `r`, `1/r`; choosing those inside the unit circle gives the minimum-phase filter `db{p}`.

**Input.** `p` (`int`, at least 1).

**Output.** `float64` array of length `2p`, equal to `pywt.Wavelet(f"db{p}").rec_lo` (`p = 1` gives Haar). Raises `ValueError` for `p < 1`.

**How to write it yourself.**
1. Check `p >= 1`.
2. Express `y = sin^2(ω/2) = (2 - z - 1/z) / 4`, i.e. the polynomial coefficients `[-0.25, 0.5, -0.25]` (powers `z^1, z^0, z^{-1}`).
3. Build `z^{p-1} P(y)` as a polynomial in `z` of degree `2(p-1)`: start from `poly = np.zeros(2p - 1)`; for each `k = 0, …, p-1`, the term `C(p-1+k, k) y^k` is the constant `comb(p-1+k, k)` convolved `k` times with `[-0.25, 0.5, -0.25]` (length `2k+1`); centre it with `np.pad(term, p - 1 - k)` and add it to `poly`.
4. Roots: `roots = np.roots(poly)` (none if `p == 1`); keep `inside = roots[np.abs(roots) < 1]`.
5. Build the filter as a polynomial product: start with `[1]`, convolve `p` times with `[1, 1]` (the roots at `-1`), then for every root `r` in `inside` convolve with `[1, -r]`.
6. Take the real part (imaginary parts cancel up to rounding) and normalize so the sum is `sqrt(2)`: `real * np.sqrt(2) / real.sum()`.

#### `mra.wavelet_filters(name)`

**What it is.** Returns the reconstruction filters of a PyWavelets wavelet, to compare own constructions with the library.

**Input.** `name` (`str`): e.g. `"haar"`, `"db3"`, `"sym4"`.

**Output.** Tuple `(h, g)` of `float64` arrays: `rec_lo` (low-pass) and `rec_hi` (high-pass).

**How to write it yourself.**
1. `wavelet = pywt.Wavelet(name)`.
2. Return `np.asarray(wavelet.rec_lo)`, `np.asarray(wavelet.rec_hi)`.

#### `mra.quadrature_mirror(h)`

**What it is.** The high-pass (wavelet) filter of an orthogonal MRA, `g[n] = (-1)^n h[L - 1 - n]`: the low-pass filter reversed, with alternating signs. With it, `ψ(x) = sqrt(2) sum_n g[n] φ(2x - n)` is the wavelet. For PyWavelets filters it equals `rec_hi`.

**Input.** `h` (array-like): low-pass filter of even length `L`.

**Output.** `float64` array of length `L`.

**How to write it yourself.**
1. `low = np.asarray(h, dtype=np.float64)`.
2. Signs `(-1.0) ** np.arange(L)`.
3. Return `signs * low[::-1]`.

#### `mra.cascade(h, iterations=8)`

**What it is.** The cascade algorithm: approximates the scaling function `φ` and the wavelet `ψ` of a filter by iterating the scaling equation `φ_{j+1}(x) = sqrt(2) sum_n h[n] φ_j(2x - n)` starting from a unit impulse. After `J` iterations the values approximate `φ` on the dyadic grid `x = m / 2^J`; `φ` is supported on `[0, L - 1]`.

**Input.** `h` (array-like): low-pass filter of length `L`; `iterations` (`int`, default 8): number of refinements `J`.

**Output.** Tuple `(x, phi, psi)` of arrays on the grid `x = m / 2^J`.

**How to write it yourself.**
1. `g = quadrature_mirror(h)`; `values = np.array([1.0])`.
2. Repeat `J` times: upsample by inserting a zero between consecutive values (`up = np.zeros(2 * len(values) - 1)`, `up[::2] = values`), then `values = np.sqrt(2) * np.convolve(up, h)`.
3. `step = 2 ** J`; `phi = values` and `x = np.arange(len(phi)) / step`.
4. Wavelet from `ψ(x) = sqrt(2) sum_n g[n] φ(2x - n)`: upsample `g` by `step` (`g_up = np.zeros((L - 1) * step + 1)`, `g_up[::step] = g`) and compute `psi = np.sqrt(2) * np.convolve(phi, g_up)`; its sample `j` belongs to `x = j / (2 step)`.
5. Interpolate `psi` onto the grid of `phi` with `np.interp(x, np.arange(len(psi)) / (2 * step), psi)` and return `x, phi, psi`.

#### `mra.packet_energies(x, wavelet="db2", level=3)`

**What it is.** Wavelet packet decomposition: unlike the ordinary DWT, which splits only the low-pass part again and again, packets split **every** band, giving `2^level` frequency bands of equal width. The function returns the energy (sum of squared coefficients) of each band, ordered from low to high frequency.

**Input.** `x` (array-like): 1D signal; `wavelet` (`str`); `level` (`int`).

**Output.** `dict` mapping node paths (strings of `a` = approximation and `d` = detail, e.g. `"aaa"`, `"aad"`, …) in frequency order to energies (`float`). For an orthogonal wavelet the energies sum to `sum(x**2)`.

**How to write it yourself.**
1. Build the tree: `tree = pywt.WaveletPacket(x, wavelet, mode="periodization", maxlevel=level)`.
2. Get the nodes of the last level in frequency order: `tree.get_level(level, order="freq")`.
3. Return `{node.path: float(np.sum(node.data ** 2)) for node in nodes}`.